In [29]:
# --- Library Imports ---
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import chi2_contingency

# --- Qiskit Imports ---
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.circuit.library import ZZFeatureMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute

print("Libraries imported successfully.")

Libraries imported successfully.


In [30]:
# Load data first
lung_cancer_column_names = ['label'] + [f'attr_{i}' for i in range(1, 57)]
file_path_lung = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\lung+cancer\lung-cancer.data'

# reads the data, treating "?" as missing values
df_lung = pd.read_csv(file_path_lung, header=None, names=lung_cancer_column_names, na_values=['?'])
print(f"Original shape of Lung Cancer data: {df_lung.shape}")

Original shape of Lung Cancer data: (32, 57)


In [31]:
# Mode imputation for missing values
modes = df_lung.mode().iloc[0]
df_lung.fillna(modes, inplace=True)

# Then check if all Nan are gone
print(f"Total missing values after imputation: {df_lung.isnull().sum().sum()}\n")

Total missing values after imputation: 0



In [32]:
# Target Binarization
df_lung['label_binary'] = df_lung['label'].apply(lambda x: 0 if x == 1 else 1)

In [33]:
# Check shape again
print("Class distribution (binary):")
print(df_lung['label_binary'].value_counts(normalize=True))

Class distribution (binary):
label_binary
1    0.71875
0    0.28125
Name: proportion, dtype: float64


In [34]:
# Separate Features & Target and Split Data
X_lung = df_lung.drop(['label', 'label_binary'], axis=1)
y_lung_binary = df_lung['label_binary']

In [35]:
X_train_lc, X_test_lc, y_train_lc, y_test_lc = train_test_split(
    X_lung, y_lung_binary, test_size=0.3, random_state=42, stratify=y_lung_binary
)

print(f"Train size: {X_train_lc.shape[0]}, Test size: {X_test_lc.shape[0]}")
print("Train class ratio:", y_train_lc.value_counts(normalize=True).to_dict())

Train size: 22, Test size: 10
Train class ratio: {1: 0.7272727272727273, 0: 0.2727272727272727}


In [36]:
# One-Hot Encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_lc_encoded = pd.DataFrame(encoder.fit_transform(X_train_lc),
columns=encoder.get_feature_names_out())
X_test_lc_encoded = pd.DataFrame(encoder.transform(X_test_lc),
columns=encoder.get_feature_names_out())

In [37]:
# Feature Selection - Cramer's V
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    if min((kcorr-1), (rcorr-1)) == 0: return 0
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

cramers_scores = {col: cramers_v(X_train_lc_encoded[col], y_train_lc) for col in X_train_lc_encoded.columns}
cramers_series = pd.Series(cramers_scores).sort_values(ascending=False)

N_FEATURES_TO_SELECT = 10 
top_features = cramers_series.head(N_FEATURES_TO_SELECT).index.tolist()

# Final Dataframes
X_train_lc_final = X_train_lc_encoded[top_features].to_numpy()
X_test_lc_final = X_test_lc_encoded[top_features].to_numpy()
y_train = y_train_lc.to_numpy()
y_test = y_test_lc.to_numpy()

print("--- Data Preprocessing Complete ---")
print(f"Final training data shape: {X_train_lc_final.shape}")
print(f"Final testing data shape: {X_test_lc_final.shape}\n")


--- Data Preprocessing Complete ---
Final training data shape: (22, 10)
Final testing data shape: (10, 10)



In [38]:
# Authenticate and list available backends
try:
    service = QiskitRuntimeService(channel="ibm_quantum_platform")
    
    # List all available backends
    print("Available backends:")
    backends = service.backends(operational=True, simulator=False)
    for b in backends:
        print(f"  - {b.name}: {b.num_qubits} qubits, status={b.status().status_msg}")
    
    # Get the least busy backend
    backend = service.least_busy(operational=True, simulator=False)
    print(f"\nSelected backend: {backend.name}")
    print(f"Number of qubits: {backend.num_qubits}")
except Exception as e:
    print("Error connecting. Please check your API token.")
    print(e)

Available backends:
  - ibm_fez: 156 qubits, status=active
  - ibm_marrakesh: 156 qubits, status=active
  - ibm_torino: 133 qubits, status=active

Selected backend: ibm_fez
Number of qubits: 156


In [39]:
# Optional: Manually select a specific backend if the least_busy one has issues
# Uncomment and modify if needed:
# backend = service.backend("ibm_brisbane")

In [40]:
# --- Create the Kernel Circuit Template ---
from qiskit.circuit import ParameterVector

feature_dim = X_train_lc_final.shape[1]

# 1. Create two feature maps with distinct parameter names
fm_x = ZZFeatureMap(feature_dimension=feature_dim, reps=2, entanglement='linear', parameter_prefix='x')
fm_y = ZZFeatureMap(feature_dimension=feature_dim, reps=2, entanglement='linear', parameter_prefix='y')

# 2. Compose them: U(x) multiplied by U(y)_inverse
kernel_circuit = fm_x.compose(fm_y.inverse())
kernel_circuit.measure_all()

print("Abstract Kernel Circuit created.")
print(f"Number of parameters: {kernel_circuit.num_parameters}")

# 3. Transpile the circuit template
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_kernel_circuit = pm.run(kernel_circuit)

print("Kernel Circuit transpiled and ISA validated.")
print(f"Hardware instruction count: {isa_kernel_circuit.count_ops()}")

Abstract Kernel Circuit created.
Number of parameters: 20
Kernel Circuit transpiled and ISA validated.
Hardware instruction count: OrderedDict([('rz', 428), ('sx', 176), ('cz', 70), ('measure', 10), ('barrier', 1)])


In [41]:
# --- Execute on Hardware with Retry Logic ---

BATCH_SIZE = 20  # Circuits per job
MAX_RETRIES = 3  # Retry on transient errors
RETRY_DELAY = 30  # Seconds between retries

def run_batch_with_retry(pubs, backend, batch_info=""):
    """Run a batch of PUBs with retry logic for transient errors."""
    for attempt in range(MAX_RETRIES):
        try:
            sampler = Sampler(mode=backend)
            job = sampler.run(pubs)
            print(f"    {batch_info} Job ID: {job.job_id()} - Waiting...")
            result = job.result()
            return result
        except Exception as e:
            error_msg = str(e)
            if "9703" in error_msg or "Temporary" in error_msg:
                print(f"    Transient error (attempt {attempt+1}/{MAX_RETRIES}): {error_msg[:50]}...")
                if attempt < MAX_RETRIES - 1:
                    print(f"    Waiting {RETRY_DELAY}s before retry...")
                    time.sleep(RETRY_DELAY)
                else:
                    raise
            else:
                raise

def run_kernel_job(X_1, X_2, parameterized_qc, backend):
    n1, n2 = len(X_1), len(X_2)
    total_evals = n1 * n2
    print(f"Preparing {n1}x{n2} = {total_evals} evaluations...")
    
    # Get the parameter objects from the circuit
    params = list(parameterized_qc.parameters)
    
    # Build all circuits by binding parameters
    all_bound_circuits = []
    for i in range(n1):
        for j in range(n2):
            param_values = np.concatenate((X_1[i], X_2[j]))
            param_dict = dict(zip(params, param_values))
            bound_qc = parameterized_qc.assign_parameters(param_dict)
            all_bound_circuits.append(bound_qc)
    
    print(f"Created {len(all_bound_circuits)} bound circuits.")
    
    # Process in batches
    all_counts = []
    num_batches = (total_evals + BATCH_SIZE - 1) // BATCH_SIZE
    
    for batch_idx in range(num_batches):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, total_evals)
        batch_circuits = all_bound_circuits[start:end]
        
        print(f"  Batch {batch_idx+1}/{num_batches}: {len(batch_circuits)} circuits...")
        
        # Create PUBs
        pubs = [(qc,) for qc in batch_circuits]
        
        result = run_batch_with_retry(pubs, backend, f"Batch {batch_idx+1}")
        
        # Each PUB result is separate
        for k in range(len(batch_circuits)):
            pub_result = result[k].data.meas
            counts = pub_result.get_counts()
            all_counts.append(counts)
    
    # Extract Kernel Values (Prob of '00...0')
    kernel_matrix = np.zeros((n1, n2))
    zero_string = '0' * parameterized_qc.num_clbits
    
    idx = 0
    for i in range(n1):
        for j in range(n2):
            counts = all_counts[idx]
            total_shots = sum(counts.values())
            zero_count = counts.get(zero_string, 0)
            kernel_matrix[i, j] = zero_count / total_shots
            idx += 1
            
    return kernel_matrix

# Execute
train_kernel = None
test_kernel = None

try:
    print("--- 1/2 Computing Training Kernel ---")
    train_kernel = run_kernel_job(X_train_lc_final, X_train_lc_final, isa_kernel_circuit, backend)
    
    print("\n--- 2/2 Computing Testing Kernel ---")
    test_kernel = run_kernel_job(X_test_lc_final, X_train_lc_final, isa_kernel_circuit, backend)
    
    print("\nSuccess! Kernel matrices ready.")
    
except Exception as e:
    print("\n!!! Execution Failed !!!")
    print(e)
    print("\nTip: If IBM backend is having issues, try:")
    print("1. Wait a few minutes and re-run this cell")
    print("2. Select a different backend (uncomment the cell above)")
    print("3. Check IBM Quantum status: https://quantum.ibm.com/")

--- 1/2 Computing Training Kernel ---
Preparing 22x22 = 484 evaluations...
Created 484 bound circuits.
  Batch 1/25: 20 circuits...
    Batch 1 Job ID: d4spv1sfitbs739j87ag - Waiting...
  Batch 2/25: 20 circuits...
    Batch 2 Job ID: d4spv9kfitbs739j87j0 - Waiting...
  Batch 3/25: 20 circuits...
    Batch 3 Job ID: d4spvhcfitbs739j87s0 - Waiting...
  Batch 4/25: 20 circuits...
    Batch 4 Job ID: d4spvpbher1c73bdjr0g - Waiting...
  Batch 5/25: 20 circuits...
    Batch 5 Job ID: d4sq0745fjns73d2o9j0 - Waiting...
  Batch 6/25: 20 circuits...
    Batch 6 Job ID: d4sq0f4fitbs739j88v0 - Waiting...
  Batch 7/25: 20 circuits...
    Batch 7 Job ID: d4sq0n4fitbs739j8990 - Waiting...
  Batch 8/25: 20 circuits...
    Batch 8 Job ID: d4sq1fjher1c73bdjsp0 - Waiting...
  Batch 9/25: 20 circuits...
    Batch 9 Job ID: d4sq1nkfitbs739j8afg - Waiting...
  Batch 10/25: 20 circuits...
    Batch 10 Job ID: d4sq26jher1c73bdjtl0 - Waiting...
  Batch 11/25: 20 circuits...
    Batch 11 Job ID: d4sq2erher1c73

KeyboardInterrupt: 

In [ ]:
# --- Hyperparameter Tuning ---
if train_kernel is not None:
    print("--- Starting Hyperparameter Tuning (Classical CPU) ---")
    
    param_grid = {
        'C': [0.1, 1, 10, 100],
        'class_weight': ['balanced', None]
    }
    
    # Use 'precomputed' kernel
    svc = SVC(kernel='precomputed')
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    grid_search = GridSearchCV(
        estimator=svc,
        param_grid=param_grid,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    # Fit using the matrix
    grid_search.fit(train_kernel, y_train)
    
    best_model = grid_search.best_estimator_
    print(f"\nBest Parameters: {grid_search.best_params_}")
    print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")
else:
    print("Skipping tuning because kernel computation failed.")

In [ ]:
# --- Final Evaluation ---
if train_kernel is not None and test_kernel is not None:
    print(f"--- Final Evaluation on Real Hardware Data ---")

    # 1. Predict
    # For precomputed kernel, we pass the Test-vs-Train matrix
    y_test_pred = best_model.predict(test_kernel)

    # 2. Calculate Metrics
    test_accuracy = accuracy_score(y_test, y_test_pred)

    print(f"Final Test Accuracy: {test_accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_test_pred, zero_division=0))

    # Optional: Visualize the Kernel Matrix
    plt.figure(figsize=(8, 6))
    plt.imshow(train_kernel, cmap='viridis')
    plt.title(f"Quantum Kernel Matrix (Real Hardware: {backend.name})")
    plt.colorbar()
    plt.show()
else:
    print("Cannot evaluate - kernel computation failed earlier.")